In [6]:
import pandas as pd
from collections import defaultdict
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [5]:
raw_counts_sarcopenia = '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/public_datasets/sarcopenia_bulk_RNA/GSE226151_SKELETAL_MUSCLE_READ_COUNT.txt'
series_matrix_sarcopenia = '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/public_datasets/sarcopenia_bulk_RNA/GSE226151_series_matrix.txt'
# read txt into a df
sarcopenia_counts = pd.read_csv(raw_counts_sarcopenia, sep='\t')
display(sarcopenia_counts)

,Name,HA_1,HA_10,HA_11,HA_12,HA_13,HA_14,HA_15,HA_16,HA_17,...,S_19,S_2,S_20,S_3,S_4,S_5,S_6,S_7,S_8,S_9
0,A1BG,0,0,0,2,1,0,2,0,0,...,2,0,1,5,1,0,0,0,0,0
1,A1CF,1,0,0,1,0,1,4,0,1,...,0,0,0,3,1,0,0,0,2,6
2,A2M,3037,2846,2604,3904,3604,5463,4026,6762,3868,...,3256,3694,2742,10628,3064,3552,2630,2990,5944,4441
3,A2ML1,0,0,0,0,1,0,0,1,0,...,0,1,0,0,1,0,0,0,0,0
4,A3GALT2,1,0,0,0,0,0,0,0,2,...,1,0,0,0,0,0,2,0,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19345,ZYG11A,2,1,1,0,0,0,0,2,0,...,0,1,4,0,0,1,0,0,1,1
19346,ZYG11B,696,1585,1102,1306,1362,1637,2867,2724,1549,...,1385,1415,3307,1705,2809,2215,1188,1730,1211,1694
19347,ZYX,524,305,291,413,391,757,331,377,432,...,553,443,610,1629,313,427,415,327,644,477
19348,ZZEF1,840,673,455,416,439,554,659,707,737,...,416,626,1036,1155,847,634,509,676,669,585


In [20]:
with open(series_matrix_sarcopenia, 'r') as f:
    lines = f.readlines()

# Step 2: Extract metadata lines
metadata = defaultdict(list)
for line in lines:
    if line.startswith("!Sample_"):
        parts = line.strip().split("\t")
        key = parts[0][8:]  # Remove "!Sample_"
        values = parts[1:]
        metadata[key].append(values)
    elif line.startswith("!series_matrix_table_begin"):
        break

# Step 3: Merge duplicate keys
merged_metadata = {}
for key, val_lists in metadata.items():
    merged_metadata[key] = list(map(lambda x: " || ".join(x), zip(*val_lists)))

# Step 4: Get sample IDs and characteristics
sample_ids = merged_metadata.get("title", [f"Sample_{i+1}" for i in range(len(next(iter(merged_metadata.values()))))])
characteristics = merged_metadata["characteristics_ch1"]

# Step 5: Parse `characteristics_ch1` into structured fields
def extract_characteristics(char_str):
    fields = char_str.split(" || ")
    parsed = {}
    for item in fields:
        if ": " in item:
            key, val = item.split(": ", 1)
            parsed[key.strip().lower()] = val.strip()
    return parsed

parsed = [extract_characteristics(c) for c in characteristics]
char_df = pd.DataFrame(parsed)
# Clean up column names: remove quotes, lowercase, and strip spaces
char_df.columns = [col.strip().strip('"').lower() for col in char_df.columns]
char_df["sample_id"] = sample_ids

# Step 6: Clean fields for labeling
char_df["sex"] = char_df["sex"].str.strip().str.strip('"').str.capitalize()
char_df["disease_state"] = char_df["disease state"].str.strip().str.strip('"').str.lower()

import re

def extract_label_from_id(sample):
    match = re.search(r'([A-Z]+)\s?0*(\d+)', sample)
    if match:
        return f"{match.group(1)}_{int(match.group(2))}"
    return "UNKNOWN"

char_df["label"] = char_df["sample_id"].str.strip().str.strip('"').apply(extract_label_from_id)

# Step 7: Map disease state to short label prefix
state_to_prefix = {
    "healthy aged": "HA",
    "sarcopenia": "S",
    "pre-sarcopenia": "PS"
}
# Step 9: Final mapping DataFrame
mapping_df = char_df[["sex", "disease_state", "label"]]

In [21]:
mapping_df

,sex,disease_state,label
0,Female,healthy aged,HA_1
1,Female,healthy aged,HA_2
2,Female,healthy aged,HA_3
3,Female,healthy aged,HA_4
4,Female,healthy aged,HA_5
5,Female,healthy aged,HA_6
6,Female,healthy aged,HA_7
7,Female,healthy aged,HA_8
8,Female,healthy aged,HA_9
9,Female,healthy aged,HA_10
